In [1]:
    module KEBProblem

        using LinearAlgebra
        using BSplineKit
        using Plots

        export KEB_LST_all,KEB_LST_OS,KEB_LST_Rayleigh,rayleigh_quotient_iteration
        function  KEB_LST_all(baseflow,N,α,β,R)
            #chebyshev配点法离散
                θ = range(0,length=N+1,stop=pi)
                x = reshape(-cos.(θ), N+1, 1)
                c = [2; ones(N-1, 1) ; 2] .* (-1) .^ (0:N)
                X = repeat(x, 1, N+1);
                dX = X - X';
                D = (c * (1 ./ c)') ./ (dX .+ I(N+1));   # off-diagonal entries
                D = D - diagm(vec(sum(D, dims=2)));      # diagonal entries  
                #interpolation
                f = open( baseflow, "r" )
                n = countlines( f )
                seekstart( f )
                data = zeros(n,6)
                for i = 1:n
                    z,w,u,v,du,dv = split( readline( f ), " " )  # 读取每一行数据并用split函数将数据“剥离”开来
                    data[i,1] = parse(Float64,z)
                    data[i,2] = parse(Float64,w)
                    data[i,3] = parse(Float64,u)
                    data[i,4] = parse(Float64,v)  # 将字符串转化为数字
                    data[i,5] = parse(Float64,du)
                    data[i,6] = parse(Float64,dv)
                end

                close( f )

                t=data[:,1]
                w=data[:,2]
                u=data[:,3]
                v=data[:,4]
                du=data[:,5]
                dv=data[:,6]
                itpw=itp = interpolate(t, w , BSplineOrder(4))
                itpu=itp = interpolate(t, u , BSplineOrder(4))
                itpv=itp = interpolate(t, v , BSplineOrder(4))
                itpdu=itp = interpolate(t, du , BSplineOrder(4))
                itpdv=itp = interpolate(t, dv , BSplineOrder(4))
                #interpolation
                # for i=1:N+1
                #     x[i,1]=5* x[i,1]+ 5
                # end
                # D=0.01* D
                # D2=D^2
                for i=1:N+1
                    D[i,:]=D[i,:].*((2*x[i]^3-x[i]^2+3*x[i]-4)^2/(20*(6*x[i]^2-2*x[i]+3)))
                end
                for i=1:N+1
                    x[i]=(4*x[i]^3-2*x[i]^2+6*x[i]+12)/(-2*x[i]^3+x[i]^2-3*x[i]+4)
                    if x[i]>10
                        x[i]=10
                    end
                end
                D2=D^2;
                D4=D^4;
                U=zeros(N+1,1)
                V=zeros(N+1,1)
                W=zeros(N+1,1)
                dU=zeros(N+1,1)
                dV=zeros(N+1,1)
                dW=zeros(N+1,1)
                p=zeros(N+1,1)
                for i=1:N+1
                    U[i,1]=itpu(x[i])
                    V[i,1]=itpv(x[i])
                    W[i,1]=itpw(x[i])
                    dU[i,1]=itpdu(x[i])
                    dV[i,1]=itpdv(x[i])
                end
                dW=-2*U
                ddU=D*dU;
                ddV=D*dV;
                α_bar=α-im/R
                λ=sqrt(α^2+β^2)
                λ_bar=sqrt(α*α_bar+β^2)
                A11=im*((D2-(λ^2)*I(N+1))*(D2-(λ_bar^2)*I(N+1)))+R*((α*U+β*V)).*(D2-(λ_bar^2)*I(N+1))-R*(α_bar*ddU+β*ddV).*I(N+1)-im*W.*D*(D^2-(λ_bar^2)*I(N+1))-im*dW.*(D2-(λ_bar^2)*I(N+1))-im*U.*D2
                A12=2*(V.+1).*D+2*dV.*I(N+1)
                A21=2*(V.+1).*D-im*R*(α*dV-β*dU).*I(N+1)
                A22=im*(D2-(λ^2)*I(N+1))+R*(α*U+β*V).*I(N+1)-im*W.*D-im*U.*I(N+1)
                B11=R*(D2-(λ_bar^2)*I(N+1))
                B12=zeros(N+1,N+1)
                B21=B12
                B22=R*I(N+1)
                B22=Matrix(B22)
                #boundary condition
                A11[2,:]=D[1,:]
                A12[2,:].=0
                A21[N,:]=D[N+1,:]
                A22[N,:].=0
                B11[2,:]=B12[2,:].=0
                B21[N,:]=B22[N,:].=0
                A11=A11[2:N,2:N]
                A12=A12[2:N,2:N]
                A21=A21[2:N,2:N]
                A22=A22[2:N,2:N]
                B11=B11[2:N,2:N]
                B12=B12[2:N,2:N]
                B21=B21[2:N,2:N]
                B22=B22[2:N,2:N]
                A=[A11 A12;A21 A22]
                B=[B11 B12;B21 B22]
                return A,B,x,D
        end
        function KEB_LST_OS(baseflow,N,α,β_bar,R)
                θ = range(0,length=N+1,stop=pi)
                x = reshape(-cos.(θ), N+1, 1)
                c = [2; ones(N-1, 1) ; 2] .* (-1) .^ (0:N)
                X = repeat(x, 1, N+1);
                dX = X - X';
                D = (c * (1 ./ c)') ./ (dX .+ I(N+1));   # off-diagonal entries
                D = D - diagm(vec(sum(D, dims=2)));      # diagonal entries
                #interpolation
                f = open( baseflow, "r" )
                n = countlines( f )
                seekstart( f )
                data = zeros(n,6)
                for i = 1:n
                    z,w,u,v,du,dv = split( readline( f ), " " )  # 读取每一行数据并用split函数将数据“剥离”开来
                    data[i,1] = parse(Float64,z)
                    data[i,2] = parse(Float64,w)
                    data[i,3] = parse(Float64,u)
                    data[i,4] = parse(Float64,v)  # 将字符串转化为数字
                    data[i,5] = parse(Float64,du)
                    data[i,6] = parse(Float64,dv)
                end

                close( f )

                t=data[:,1]
                w=data[:,2]
                u=data[:,3]
                v=data[:,4]
                du=data[:,5]
                dv=data[:,6]
                itpw=itp = interpolate(t, w , BSplineOrder(4))
                itpu=itp = interpolate(t, u , BSplineOrder(4))
                itpv=itp = interpolate(t, v , BSplineOrder(4))
                itpdu=itp = interpolate(t, du , BSplineOrder(4))
                itpdv=itp = interpolate(t, dv , BSplineOrder(4))
                psi=zeros(N+1,1)
                for i=1:N+1
                    psi[i,1]=0.6*x[i,1]+0.4*(x[i,1]^3+0.5*(1-x[i,1]^2))
                    x[i,1]=2*(1+psi[i,1])/(1-psi[i,1])
                end
                D=0.2*D
                D2=D^2
                U=zeros(N+1,1)
                V=zeros(N+1,1)
                W=zeros(N+1,1)
                dU=zeros(N+1,1)
                dV=zeros(N+1,1)
                dW=zeros(N+1,1)
                p=zeros(N+1,1)
                for i=1:75
                    U[i,1]=itpu(x[i])
                    V[i,1]=itpv(x[i])
                    W[i,1]=itpw(x[i])
                    dU[i,1]=itpdu(x[i])
                    dV[i,1]=itpdv(x[i])
                end
                for i=76:100
                    U[i,1]=U[75,1]
                    V[i,1]=V[75,1]
                    W[i,1]=W[75,1]
                    dU[i,1]=dU[75,1]
                    dV[i,1]=dV[75,1]
                end
                for i=1:N
                    x[i,1]=0.2*(x[i,1]-5)
                end
                dW=-2*U
                ddU=D*dU;
                ddV=D*dV;
                λ=sqrt(α^2+β_bar^2)
                A=(D2-(λ^2)*I(N+1))^2-im*R*(α*U+β_bar*V).*(D2-(λ^2)*I(N+1))+im*R*(α*ddU+β_bar*ddV).*I(N+1)
                B=-1*im*R*α*(D2-(λ^2)*I(N+1))
                A[2,:]=D[1,:]
                A[N,:]=D[N+1,:]
                B[2,:]=B[N,:].=0
                A=A[2:N,2:N]
                B=B[2:N,2:N]
                return A,B,x
        end
        function KEB_LST_Rayleigh(baseflow,N,α,β)



                θ = range(0,length=N+1,stop=pi)
                x = reshape(-cos.(θ), N+1, 1)
                c = [2; ones(N-1, 1) ; 2] .* (-1) .^ (0:N)
                X = repeat(x, 1, N+1);
                dX = X - X';
                D = (c * (1 ./ c)') ./ (dX .+ I(N+1));   # off-diagonal entries
                D = D - diagm(vec(sum(D, dims=2)));      # diagonal entries
                #interpolation
                f = open( baseflow, "r" )
                n = countlines( f )
                seekstart( f )
                data = zeros(n,6)
                for i = 1:n
                    z,w,u,v,du,dv = split( readline( f ), " " )  # 读取每一行数据并用split函数将数据“剥离”开来
                    data[i,1] = parse(Float64,z)
                    data[i,2] = parse(Float64,w)
                    data[i,3] = parse(Float64,u)
                    data[i,4] = parse(Float64,v)  # 将字符串转化为数字
                    data[i,5] = parse(Float64,du)
                    data[i,6] = parse(Float64,dv)
                end

                close( f )

                t=data[:,1]
                w=data[:,2]
                u=data[:,3]
                v=data[:,4]
                du=data[:,5]
                dv=data[:,6]
                itpw=itp = interpolate(t, w , BSplineOrder(4))
                itpu=itp = interpolate(t, u , BSplineOrder(4))
                itpv=itp = interpolate(t, v , BSplineOrder(4))
                itpdu=itp = interpolate(t, du , BSplineOrder(4))
                itpdv=itp = interpolate(t, dv , BSplineOrder(4))
                psi=zeros(N+1,1)
                for i=1:N+1
                    x[i,1]=5*x[i,1]+5
                end
                D=0.2*D
                D2=D^2
                U=zeros(N+1,1)
                V=zeros(N+1,1)
                W=zeros(N+1,1)
                dU=zeros(N+1,1)
                dV=zeros(N+1,1)
                dW=zeros(N+1,1)
                p=zeros(N+1,1)
                for i=1:N+1
                    U[i,1]=itpu(x[i])
                    V[i,1]=itpv(x[i])
                    W[i,1]=itpw(x[i])
                    dU[i,1]=itpdu(x[i])
                    dV[i,1]=itpdv(x[i])
                end
                for i=1:N
                    x[i,1]=0.2*(x[i,1]-5)
                end
                dW=-2*U
                ddU=D*dU;
                ddV=D*dV;
                λ=sqrt(α^2+β^2)
                A=(α*U+β*V).*(D2-(λ^2)*I(N+1))-(α*ddU+β*ddV).*I(N+1)
                B=D2-(λ^2)*I(N+1)
                A[N,:]=D[N+1,:]
                B[2,:]=B[N,:].=0
                A=A[2:N,2:N]
                B=B[2:N,2:N]
                return A,B,x
            
        end
    end
    function rayleigh_quotient_iteration(A, B, sigma; q0=rand(size(A, 1), 1))

        flg = true
        while flg
            sigma0 = sigma[1] + 0.0e0im
            q = (A - sigma*B) \ (B*q0)
            q0 = q/maximum(abs.(q))
            sigma = ((q0'*(A*q0))/(q0'*(B*q0)))[1]
            if abs(sigma-sigma0)<=eps(1.0f0)
                flg = false
            end
        end

        return sigma, q0

    end

rayleigh_quotient_iteration (generic function with 1 method)

In [6]:
using .KEBProblem
using LinearAlgebra
N=299
α=0.38482
β=0.07759
R=286
# R=440.88
# α=0.13228
# β=0.04367
A,B,x,D=KEB_LST_all("Vonkarmen.txt",N,α,β,R)
A=A[2:end-1,2:end-1]
B=B[2:end-1,2:end-1]
@time C=eigen(A,B)
values=C.values
vectors=C.vectors
K=filter(x-> 0<imag(x)<1,values)
K1=map(x->0<imag(x)<0.1,values)
K2=findmax(K1)
index=K2[2]
c_temp=values[index,1]
q1=vectors[:,index]
@time c,q=rayleigh_quotient_iteration(A,B,c_temp,q0=q1)
c

  2.226478 seconds (19 allocations: 16.538 MiB)
  0.033864 seconds (35 allocations: 32.415 MiB, 23.81% gc time)


-6.397535386350755e-6 + 2.866796334403976e-5im

In [ ]:
using .KEBProblem
c0=c
q0=q
c_temp=c0
q_temp=q
c_all=nothing
N=199
R=400
index_α=nothing
index_β=nothing
for α=0.01:0.01:0.8
    @show α
    A,B=KEB_LST_all("Vonkarmen copy 2.txt",N,α,0.101,R)
    G=eigen(A,B)
    values=G.values
    K=filter(x-> 1>imag(x)>-2,values)
    K1=findmax(imag,K)
    index=K1[2]
    c_temp=K[index,1]
    if c_temp==-Inf
        c_temp=0
    end
    q_temp=vectors[:,index]
    for β=0.101:0.001:0.4
        A,B=KEB_LST_all("Vonkarmen copy 2.txt",N,α,β,R)
        c_temp,q_temp=rayleigh_quotient_iteration(A,B,c_temp,q0=q_temp)
        c_all=[c_all;c_temp]
        index_α=[index_α;α]
        index_β=[index_β;β]
    end
end

In [ ]:
#导入get到的文献数据
f = open( "hr.dat", "r" )
n = countlines( f )
seekstart( f )
data_hr = zeros(n,2)
for i = 1:n
    z_hr,hr= split( readline( f ) )  # 读取每一行数据并用split函数将数据“剥离”开来
    data_hr[i,1] = parse(Float64,z_hr)
    data_hr[i,2] = parse(Float64,hr)
end

close( f )
f = open( "hi.dat", "r" )
n = countlines( f )
seekstart( f )
data_hi = zeros(n,2)
for i = 1:n
    z_hi,hi = split( readline( f ))  # 读取每一行数据并用split函数将数据“剥离”开来
    data_hi[i,1] = parse(Float64,z_hi)
    data_hi[i,2] = parse(Float64,hi)
end

close( f )

f = open( "gr.dat", "r" )
n = countlines( f )
seekstart( f )
data_gr = zeros(n,2)
for i = 1:n
    z_gr,gr = split( readline( f ))  # 读取每一行数据并用split函数将数据“剥离”开来
    data_gr[i,1] = parse(Float64,z_gr)
    data_gr[i,2] = parse(Float64,gr)
end

close( f )

f = open( "gi.dat", "r" )
n = countlines( f )
seekstart( f )
data_gi = zeros(n,2)
for i = 1:n
    z_gi,gi = split( readline( f ))  # 读取每一行数据并用split函数将数据“剥离”开来
    data_gi[i,1] = parse(Float64,z_gi)
    data_gi[i,2] = parse(Float64,gi)
end

close( f )

f = open( "fr.dat", "r" )
n = countlines( f )
seekstart( f )
data_fr = zeros(n,2)
for i = 1:n
    z_fr,fr = split( readline( f ))  # 读取每一行数据并用split函数将数据“剥离”开来
    data_fr[i,1] = parse(Float64,z_fr)
    data_fr[i,2] = parse(Float64,fr)
end

close( f )

f = open( "fi.dat", "r" )
n = countlines( f )
seekstart( f )
data_fi = zeros(n,2)
for i = 1:n
    z_fi,fi = split( readline( f ))  # 读取每一行数据并用split函数将数据“剥离”开来
    data_fi[i,1] = parse(Float64,z_fi)
    data_fi[i,2] = parse(Float64,fi)
end

close( f )

In [ ]:
#去除0和无穷远处的点（根据边界条件等于0）
D=D[2:299,2:299]

In [ ]:
#将文献的点插值到离散点上
using BSplineKit
using LinearAlgebra
data_hi[1,1]=0
data_hr[1,1]=0
data_gi[1,1]=0
data_gr[1,1]=0
data_fi[1,1]=0
data_fr[1,1]=0
hi=data_hi[:,2]
hr=data_hr[:,2]
x_hi=data_hi[:,1]
x_hr=data_hr[:,1]
data_gi[1,1]=0
data_gr[1,1]=0
gi=data_gi[:,2]
gr=data_gr[:,2]
x_gi=data_gi[:,1]
x_gr=data_gr[:,1]
fi=data_fi[:,2]
fr=data_fr[:,2]
x_fi=data_fi[:,1]
x_fr=data_fr[:,1]
itphi = interpolate(x_hi, hi , BSplineOrder(4))
itphr = interpolate(x_hr, hr , BSplineOrder(4))
itpgi = interpolate(x_gi, gi , BSplineOrder(4))
itpgr = interpolate(x_gr, gr , BSplineOrder(4))
itpfi = interpolate(x_fi, fi , BSplineOrder(4))
itpfr = interpolate(x_fr, fr , BSplineOrder(4))
Hi=zeros(N+1,1)
Hr=zeros(N+1,1)
Gi=zeros(N+1,1)
Gr=zeros(N+1,1)
Fi=zeros(N+1,1)
Fr=zeros(N+1,1)
for i=1:N+1
    Hi[i,1]=itphi(x[i,1])
    Hr[i,1]=itphr(x[i,1])
    Gi[i,1]=itpgi(x[i,1])
    Gr[i,1]=itpgr(x[i,1])
    Fi[i,1]=itpfi(x[i,1])
    Fr[i,1]=itpfr(x[i,1])
end

In [ ]:
#求插值后的点的模
abs_H=Hi.*Hi+Hr.*Hr
abs_H=sqrt.(abs_H)
abs_F=Fi.*Fi+Fr.*Fr
abs_F=sqrt.(abs_F)
abs_G=Gi.*Gi+Gr.*Gr
abs_G=sqrt.(abs_G)


In [9]:
#处理解特征值问题得到的数据
using Plots
using BSplineKit
using LinearAlgebra
x_bsp=x[2:end-1,1]
q_real=real(q[1:298,1]  )
q_imag=imag(q[1:298,1])
dhi=D[1:298,1:298]*q_imag
dhr=D[1:298,1:298]*q_real
η=q[299:594,1]
ηr=real(η)
ηi=imag(η)
A11=α*I(N-1);A12=(1/R)*I(N-1);A13=β*I(N-1);A14=zeros(N-1,N-1)
A21=A12;A22=-1*A11;A23=A14;A24=-1*A13;
A31=A24;A32=A34=A14;A33=A11;
A41=A43=A14;A42=A31;A44=A33
A=[A11 A12 A13 A14;A21 A22 A23 A24;A31 A32 A33 A34;A41 A42 A43 A44;]
B1=-1*dhi
B2=-1*dhr
B3=ηr
B4=ηi
B=[B1;B2;B3;B4]
t=A\B
fr=t[1:298,1]
fi=t[299:2*298,1]
gr=t[2*298+1:3*298,1]
gi=t[3*298+1:4*298,1]
u=findmax(gi)
hhr=q_real
hhi=q_imag
gi=gi./u[1]
gr=gr./u[1]
fi=fi./u[1]
fr=fr./u[1]
hhr=hhr./u[1]
hhi=hhi./u[1]
abs_h=hhr.*hhr+hhi.*hhi
abs_h=sqrt.(abs_h)
abs_f=fr.*fr+fi.*fi
abs_f=sqrt.(abs_f)
abs_g=gr.*gr+gi.*gi
abs_g=sqrt.(abs_g)
y=findmax(abs_g)
abs_g=abs_g./y[1]
abs_f=abs_f./y[1]
abs_h=abs_h./y[1]
abs_f=abs_f./2


DimensionMismatch: DimensionMismatch: arguments must have the same number of rows

In [ ]:
#汇总数据
data_pa=[x abs_F abs_G abs_H]
data_ca_gh=[x_bsp abs_g abs_h]
data_ca_f=[x_bsp_f abs_f]
x_bsp_f=x_bsp.*0.78
plot(x,abs_H,xlims=[0,6])
plot!(x,abs_F,xlims=[0,6])
plot!(x,abs_G,xlims=[0,6])
plot!(x_bsp_f,abs_f)
plot!(x_bsp,abs_g)
plot!(x_bsp,abs_h)
# scatter(x,abs_G)

In [ ]:
#解Ekman边界层特征曲线
using .KEBProblem
using LinearAlgebra
N=99
α=0.2
β=0.2
c_all=nothing
index_α=nothing
index_β=nothing
for α=0.01:0.01:1
    @show α
    for β=0.001:0.001:0.4
        A,B,x=KEB_LST_Rayleigh("Ekman copy.txt",N,α,β)
        F=eigen(A,B)
        values=F.values
        Q=filter(x->1>imag(x)>0,values)
        push!(Q,0)
        K=findmax(imag(Q))
        index=K[2]
        c_all=[c_all;imag(Q[index,1])]
        index_α=[index_α;α]
        index_β=[index_β;β]
    end
end

In [ ]:
#汇总数据
c_all=c_all[2:end,1]
index_α=index_α[2:end,1]
index_β=index_β[2:end,1]
data=[index_α index_β c_all]

In [ ]:
#Output
using DelimitedFiles
writedlm("R=300.dat",data ,'\t')
# writedlm("R=285.36_pa.dat",data_pa, '\t')
# writedlm("R=285.36_ca_f.dat",data_ca_f, '\t')